# Business Entity Resolution Pipeline — Colab Runner

This notebook sets up the environment, clones the repository, links the dataset, and runs the training & inference pipeline.

## 1. Environment & GPU Check

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Clone Repository & Install Dependencies

In [ ]:
# Clone the repo
!git clone https://github.com/tamcee/amazon-ml.git
%cd amazon-ml/student_resource/code/business_entity_resolution

# Install dependencies
!pip install -q -r requirements.txt

## 3. Dataset Setup

Choose one of the methods below to make the dataset available to the pipeline:

### Option A: Mount Google Drive (Recommended if dataset is in Drive)
```python
from google.colab import drive
drive.mount('/content/drive')

# Set environment variables or symlink
import os
os.environ['TRAIN_DIR'] = '/content/drive/MyDrive/path_to_dataset/train'
os.environ['TEST_DIR'] = '/content/drive/MyDrive/path_to_dataset/test'
os.environ['OUTPUT_DIR'] = '/content/amazon-ml/student_resource/output'
```

### Option B: Download via Kaggle API / Direct link / Upload
```python
# e.g. upload kaggle.json or extract zip
# !unzip -q /content/dataset.zip -d /content/amazon-ml/student_resource/
```

In [ ]:
import os

# Example: If data is uploaded or extracted into student_resource/dataset:
# Check if default dataset folder exists
default_train = "../../dataset/train"
if os.path.exists(default_train):
    print("Default dataset directory found:", os.listdir(default_train))
else:
    print("Please configure TRAIN_DIR and TEST_DIR environment variables if dataset is mounted elsewhere.")

## 4. Run Development Cohort (Smoke Test / Quick Verification)

In [ ]:
# Run a 10,000-entity dev cohort first to verify blocking recall, feature extraction & training
!python run_pipeline.py --stage dev_cohort --dev-size 10000

# Train on the dev cohort
!TRAIN_DIR=artifacts/dev_cohort python run_pipeline.py --stage train

## 5. Run Full-Scale Training and Test Inference

In [ ]:
# Run full training on the complete dataset
!python run_pipeline.py --stage train

# Run full test inference (generates matching_results.tsv and candidate_pairs.tsv)
!python run_pipeline.py --stage infer

## 6. Validate Submission Files

In [ ]:
!python ../../utils/validate_submission.py \
    --matching ../../output/matching_results.tsv \
    --candidate ../../output/candidate_pairs.tsv \
    --test-dir ../../dataset/test

## 7. Package Outputs for Download

In [ ]:
!zip -r /content/submission_output.zip ../../output/